# 🌳 Deforestation Tracker (Full Auto Pipeline)
Includes Kaggle Download + Model + Satellite + Heatmap

## 📥 Upload Kaggle API Key

In [ ]:
from google.colab import files
files.upload()  # upload kaggle.json

## 🔐 Setup Kaggle

In [ ]:

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json


## 📦 Download Dataset

In [ ]:
!kaggle competitions download -c dsc6232-rwanda-summer2020-hw2
!unzip dsc6232-rwanda-summer2020-hw2.zip -d data

## 🔧 Convert Dataset

In [ ]:

import os, shutil, pandas as pd

os.makedirs("dataset/forest", exist_ok=True)
os.makedirs("dataset/deforested", exist_ok=True)

# MODIFY depending on dataset structure
try:
    df = pd.read_csv("data/train.csv")
    for _, row in df.iterrows():
        img = f"data/{row['image_id']}.png"
        if os.path.exists(img):
            dest = "dataset/deforested" if row['label']==1 else "dataset/forest"
            shutil.copy(img, dest)
except:
    print("Adjust dataset paths manually if needed")


## 🤖 Train Models

In [ ]:

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_dir = "dataset"

datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_data = datagen.flow_from_directory(train_dir, target_size=(224,224), batch_size=32, class_mode='binary', subset='training')
val_data = datagen.flow_from_directory(train_dir, target_size=(224,224), batch_size=32, class_mode='binary', subset='validation')


In [ ]:

from tensorflow.keras import layers, models
cnn = models.Sequential([
    layers.Conv2D(32,(3,3),activation='relu',input_shape=(224,224,3)),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128,activation='relu'),
    layers.Dense(1,activation='sigmoid')
])
cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn.fit(train_data, validation_data=val_data, epochs=3)


In [ ]:

from tensorflow.keras.applications import ResNet50
base = ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))
for l in base.layers: l.trainable=False

resnet = models.Sequential([base, layers.GlobalAveragePooling2D(), layers.Dense(1,activation='sigmoid')])
resnet.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
resnet.fit(train_data, validation_data=val_data, epochs=3)


## 📊 Compare Models

In [ ]:

import matplotlib.pyplot as plt
acc = {"CNN": cnn.evaluate(val_data)[1], "ResNet": resnet.evaluate(val_data)[1]}
plt.bar(acc.keys(), acc.values())
plt.show()


## 🛰️ Fetch Satellite

In [ ]:

import ee, requests
ee.Authenticate()
ee.Initialize()

region = ee.Geometry.Rectangle([29.5, -2.5, 30.5, -1.5])
img = ee.ImageCollection("COPERNICUS/S2").filterBounds(region).first()

url = img.getThumbURL({'region': region,'dimensions':512,'bands':['B4','B3','B2']})
r = requests.get(url)
open("sat.png","wb").write(r.content)


## 🔧 Patch Split + Predict

In [ ]:

import cv2, numpy as np, os
os.makedirs("patches", exist_ok=True)

img = cv2.imread("sat.png")
paths=[]

for i in range(0,img.shape[0],224):
    for j in range(0,img.shape[1],224):
        p = img[i:i+224,j:j+224]
        if p.shape[0]==224:
            path=f"patches/{len(paths)}.png"
            cv2.imwrite(path,p)
            paths.append(path)

results=[]
for p in paths:
    im=cv2.imread(p)
    im=cv2.resize(im,(224,224))/255.0
    im=np.reshape(im,(1,224,224,3))
    pred=resnet.predict(im)[0][0]
    results.append(1 if pred>0.5 else 0)


## 🔥 Heatmap

In [ ]:

import numpy as np
heat=np.zeros((img.shape[0],img.shape[1]))
k=0
for i in range(0,img.shape[0],224):
    for j in range(0,img.shape[1],224):
        if k<len(results):
            heat[i:i+224,j:j+224]=results[k]
            k+=1

import matplotlib.pyplot as plt
plt.imshow(heat,cmap='hot')
plt.colorbar()
plt.show()
